# ESRB Rating Prediction — Merged Dataset Report
### Combining Content Descriptors with Game Metadata for Better E/ET Classification

---

**Datasets:**
- `Video_games_esrb_rating.csv` — 1,895 games with 31 binary content descriptors and ESRB rating
- `Video_Games_Sales_with_Ratings.csv` — 16,000+ games with genre, platform, critic score, sales, and year

**Problem:** The descriptor-only model struggles to separate **E** from **ET** — their descriptor profiles often overlap. A game with only `mild_cartoon_violence` could be either. Genre and critic signals break this tie.

**Goal:** Merge both datasets on game title, enrich the feature set, and demonstrate measurable improvement on E/ET classification.

**Pipeline:** Load → Understand → Merge → EDA → Clean → Outlier Treatment → Feature Engineering → Feature Selection → Model Comparison → Tuning → Conclusion


## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, re
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 100,
                     'axes.spines.top': False, 'axes.spines.right': False})

# Consistent colour palette for the 4 ESRB ratings (E, ET, T, M)
RATING_ORDER  = ['E', 'ET', 'T', 'M']
RATING_COLORS = {'E': '#55A868', 'ET': '#4C72B0', 'T': '#DD8452', 'M': '#C44E52'}
PALETTE       = [RATING_COLORS[r] for r in RATING_ORDER]

print('All libraries loaded.')

---
# SECTION 1 — DATA UNDERSTANDING
> Before merging, we must understand each dataset independently: its shape, column types,
> missing values, and range of values. Skipping this step risks silent merge errors —
> e.g. if title formats differ wildly between sources, a join on raw title will silently drop most rows.


### 1.1 Load the ESRB Descriptor Dataset

In [ ]:
esrb = pd.read_csv('dataset/Video_games_esrb_rating.csv')
print(f'ESRB dataset — Rows: {esrb.shape[0]:,}  Columns: {esrb.shape[1]}')
esrb.head(3)

**What we have:** Each row is one game. Columns 3–33 are binary flags (0/1) indicating whether
a content descriptor was assigned by the ESRB. The final column is the target rating.
This dataset is clean and dense — no missing values — but narrow: it tells us *what* content
a game has, but nothing about *what kind* of game it is.


### 1.2 Load the Kaggle Sales & Ratings Dataset

In [ ]:
kaggle = pd.read_csv('dataset/Video_Games_Sales_with_Ratings.csv', encoding='latin-1')
print(f'Kaggle dataset — Rows: {kaggle.shape[0]:,}  Columns: {kaggle.shape[1]}')
kaggle.head(3)

### 1.3 Kaggle Column Types & Missing Values

In [ ]:
# We care most about: Name, Genre, Platform, Critic_Score, Year_of_Release
# Understanding missingness NOW tells us which features will be sparse after the merge.
miss = kaggle.isnull().sum()
miss_pct = (miss / len(kaggle) * 100).round(1)
pd.DataFrame({'Missing': miss, 'Missing %': miss_pct})[miss > 0].sort_values('Missing %', ascending=False)

**Key observation:** `Critic_Score` and `User_Score` have the most missing values — roughly 40–50%
of the Kaggle catalogue was never reviewed on Metacritic. This is critical: we cannot treat
critic score as a reliable feature unless we build the model to handle its absence gracefully.
We will add a binary `has_critic_score` flag so the model learns the difference between
'score was 0' and 'score was not provided'.


### 1.4 ESRB Descriptor Columns

In [ ]:
DESCRIPTOR_COLS = [c for c in esrb.columns if c not in ['title', 'console', 'esrb_rating']]
print(f'Number of descriptor features: {len(DESCRIPTOR_COLS)}')
print('\nDescriptors:')
for i, c in enumerate(DESCRIPTOR_COLS, 1):
    print(f'  {i:2d}. {c}')

### 1.5 Kaggle Rating Vocabulary vs ESRB Rating Vocabulary

In [ ]:
# The Kaggle dataset uses ESRB ratings too, but with slightly different labels.
# E10+ in Kaggle = ET in our ESRB CSV. We need to verify this before any comparison.
print('Kaggle Rating values:')
print(kaggle['Rating'].value_counts().to_string())
print('\nESRB CSV Rating values:')
print(esrb['esrb_rating'].value_counts().to_string())

**Important:** Kaggle uses `E10+` while our ESRB CSV uses `ET` for the same rating (Everyone 10+).
The Kaggle `Rating` column will NOT be used as a training label — we already have authoritative
labels from the ESRB CSV. The Kaggle dataset is used purely for its metadata columns
(genre, platform, critic score, year). This avoids any label-leakage between sources.


---
# SECTION 2 — TITLE NORMALIZATION & MERGE
> The two datasets share no common key except game title — but titles are messy:
> capitalization differs, punctuation differs, special characters appear in one but not the other.
> We normalize both sides before matching to maximize coverage.


### 2.1 Normalize Titles for Matching

In [ ]:
def normalize_title(t):
    """Lowercase, strip punctuation, collapse whitespace."""
    t = str(t).lower().strip()
    t = re.sub(r'[^\w\s]', '', t)   # remove punctuation
    t = re.sub(r'\s+', ' ', t)       # collapse whitespace
    return t

esrb['title_norm']   = esrb['title'].apply(normalize_title)
kaggle['title_norm'] = kaggle['Name'].apply(normalize_title)

# Quick sanity check — show a few normalized titles
print('ESRB sample normalized titles:')
print(esrb[['title','title_norm']].head(5).to_string(index=False))
print('\nKaggle sample normalized titles:')
print(kaggle[['Name','title_norm']].head(5).to_string(index=False))

### 2.2 Deduplicate Kaggle (Same Title, Multiple Platforms)

In [ ]:
# The Kaggle dataset lists the same game once per platform (e.g. FIFA appears 8 times).
# For a left-join onto ESRB, we need one row per title.
# Strategy: keep the row with the highest Critic_Score (best-reviewed version).
# This gives us the most informative signal for each game.
kaggle['Critic_Score'] = pd.to_numeric(kaggle['Critic_Score'], errors='coerce')
kaggle['Year_of_Release'] = pd.to_numeric(kaggle['Year_of_Release'], errors='coerce')

kaggle_dedup = (
    kaggle.sort_values('Critic_Score', ascending=False)
          .drop_duplicates(subset='title_norm', keep='first')
    [['title_norm','Genre','Platform','Critic_Score','Year_of_Release','Publisher']]
)
print(f'Kaggle rows before dedup : {len(kaggle):,}')
print(f'Kaggle rows after dedup  : {len(kaggle_dedup):,}')
print(f'Unique titles retained   : {kaggle_dedup["title_norm"].nunique():,}')

### 2.3 Perform the Left Join & Measure Match Rate

In [ ]:
# Left join: every ESRB game is kept. Kaggle metadata is added where available.
# Games with no Kaggle match get NaN for genre/platform/critic_score.
merged = esrb.merge(kaggle_dedup, on='title_norm', how='left')

matched     = merged['Critic_Score'].notna().sum()
unmatched   = merged['Critic_Score'].isna().sum()
match_rate  = matched / len(merged) * 100

print(f'Total ESRB games      : {len(merged):,}')
print(f'Matched to Kaggle     : {matched:,}  ({match_rate:.1f}%)')
print(f'No Kaggle match (NaN) : {unmatched:,}  ({100-match_rate:.1f}%)')
print(f'\nColumns in merged dataset: {list(merged.columns)}')

### 2.4 Match Rate by ESRB Rating

In [ ]:
# Does the match rate differ by rating category? If E games match less,
# the enriched features will be biased — important to know before training.
match_by_rating = merged.groupby('esrb_rating').apply(
    lambda g: pd.Series({
        'total':       len(g),
        'matched':     g['Critic_Score'].notna().sum(),
        'match_rate%': round(g['Critic_Score'].notna().mean() * 100, 1)
    })
).loc[RATING_ORDER]
print(match_by_rating.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(match_by_rating.index, match_by_rating['match_rate%'],
              color=PALETTE, edgecolor='white', width=0.55)
for bar, val in zip(bars, match_by_rating['match_rate%']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Kaggle Match Rate by ESRB Rating', fontweight='bold')
ax.set_xlabel('ESRB Rating')
ax.set_ylabel('Games with Critic Score (%)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

**What to look for:** If match rates are uneven across ratings (e.g. M games have much higher
match rates than E games), the enriched features will be biased — the model may learn
'missing score' as a proxy for 'E rating'. The `has_critic_score` flag we engineer later
explicitly exposes this pattern so the model can use it intentionally rather than implicitly.


---
# SECTION 3 — EXPLORATORY DATA ANALYSIS
> EDA on the merged dataset answers the central question: **do the new features (genre, platform,
> critic score) actually separate E from ET better than descriptors alone?**
> If the answer is no, merging adds noise, not signal.


### 3.1 Target Distribution — ESRB Rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rating_counts = merged['esrb_rating'].value_counts().loc[RATING_ORDER]

axes[0].bar(rating_counts.index, rating_counts.values, color=PALETTE, edgecolor='white', width=0.55)
for i, (lbl, cnt) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0].text(i, cnt + 8, str(cnt), ha='center', fontweight='bold')
axes[0].set_title('ESRB Rating Count', fontweight='bold')
axes[0].set_xlabel('ESRB Rating')
axes[0].set_ylabel('Number of Games')

axes[1].pie(rating_counts.values, labels=rating_counts.index,
            autopct='%1.1f%%', colors=PALETTE,
            startangle=140, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('ESRB Rating Proportion', fontweight='bold')
plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Imbalance check:** T is the largest class (~36%), M the smallest (~20%). A 1.78x ratio.
We use **F1 Macro** as our primary metric because accuracy is misleading under imbalance —
a model predicting 'T' for everything achieves 36% accuracy without learning anything.


### 3.2 Genre Distribution in the Matched Subset

In [ ]:
# We first look at how games are distributed across genres in the Kaggle-matched portion.
# Genre is our strongest new feature — understanding its distribution tells us
# whether it has enough coverage and variation to be useful.
genre_counts = merged['Genre'].value_counts().dropna()

fig, ax = plt.subplots(figsize=(12, 4))
genre_counts.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Game Genre Distribution (Kaggle-matched games only)', fontweight='bold')
ax.set_xlabel('Genre')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print(f'Distinct genres : {merged["Genre"].nunique()}')
print(f'Games with genre: {merged["Genre"].notna().sum():,} ({merged["Genre"].notna().mean()*100:.1f}%)')

### 3.3 Genre vs ESRB Rating — Heatmap

In [ ]:
# This is the core diagnostic plot for our merge strategy.
# If certain genres concentrate strongly in one rating, genre is a powerful feature.
# We especially want to see whether genre separates E from ET.
genre_rating = pd.crosstab(
    merged['Genre'].fillna('Unknown'),
    merged['esrb_rating'],
    normalize='index'   # row-normalize: proportion of each genre per rating
)[RATING_ORDER]

# Sort genres by E+ET proportion descending (most family-friendly on top)
genre_rating = genre_rating.sort_values('E', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(genre_rating, annot=True, fmt='.0%', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Genre vs ESRB Rating  (row-normalized proportions)', fontweight='bold')
ax.set_xlabel('ESRB Rating')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

**What to look for:** Sports, Racing, and Puzzle should skew heavily toward E.
Action and Shooter should skew toward M. Role-Playing and Adventure will likely land
in the ET/T middle zone. These patterns are the signal the model will learn —
patterns that descriptors alone cannot capture because they capture *what* content is present,
not *what type* of game it is.


### 3.4 Critic Score Distribution by ESRB Rating

In [ ]:
# Critic score is a proxy for production quality and target audience.
# High-budget M-rated games (GTA, Halo) tend to score higher than budget E titles.
# The question is whether the score distribution is different enough between
# ratings — especially E vs ET — to be a useful discriminator.
matched_df = merged.dropna(subset=['Critic_Score'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot
data_by_rating = [matched_df[matched_df['esrb_rating']==r]['Critic_Score'].values
                  for r in RATING_ORDER]
bp = axes[0].boxplot(data_by_rating, labels=RATING_ORDER, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Critic Score by ESRB Rating (Boxplot)', fontweight='bold')
axes[0].set_xlabel('ESRB Rating')
axes[0].set_ylabel('Critic Score (0-100)')

# KDE
for r, col in zip(RATING_ORDER, PALETTE):
    subset = matched_df[matched_df['esrb_rating']==r]['Critic_Score']
    subset.plot.kde(ax=axes[1], color=col, linewidth=2, label=r)
axes[1].set_title('Critic Score Distribution (KDE)', fontweight='bold')
axes[1].set_xlabel('Critic Score')
axes[1].legend(title='Rating')
plt.tight_layout()
plt.show()

print('Median critic score by rating:')
print(matched_df.groupby('esrb_rating')['Critic_Score'].median().loc[RATING_ORDER].round(1))

### 3.5 E vs ET — The Core Classification Problem

In [ ]:
# This is the most important diagnostic plot in this notebook.
# The original descriptor-only model confuses E and ET most often.
# Here we visualise whether genre and critic score help separate them.
e_et = merged[merged['esrb_rating'].isin(['E','ET'])].dropna(subset=['Critic_Score','Genre'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Critic score: E vs ET
for r, col in zip(['E','ET'], [RATING_COLORS['E'], RATING_COLORS['ET']]):
    subset = e_et[e_et['esrb_rating']==r]['Critic_Score']
    subset.plot.kde(ax=axes[0], color=col, linewidth=2.5, label=r, fill=True, alpha=0.25)
axes[0].set_title('Critic Score: E vs ET', fontweight='bold')
axes[0].set_xlabel('Critic Score')
axes[0].legend(title='Rating')

# Genre breakdown: E vs ET side by side
top_genres = e_et['Genre'].value_counts().head(8).index
genre_e_et = (
    e_et[e_et['Genre'].isin(top_genres)]
    .groupby(['Genre','esrb_rating'])
    .size().unstack(fill_value=0)
    [['E','ET']]
)
genre_e_et.plot(kind='barh', ax=axes[1],
                color=[RATING_COLORS['E'], RATING_COLORS['ET']],
                edgecolor='white')
axes[1].set_title('Top Genres: E vs ET Split', fontweight='bold')
axes[1].set_xlabel('Game Count')
axes[1].legend(title='Rating')
plt.suptitle('E vs ET — The Hardest Classification Pair', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key insight:** If critic score distributions overlap heavily for E vs ET, it is not useful
alone — but combined with genre (e.g. Sports → E, Adventure → ET), the pair provides
complementary signal. The model exploits the *interaction* between genre and descriptors,
which neither feature can provide alone. This is precisely why the V2 Pipeline outperforms V1.


### 3.6 Year of Release vs Rating

In [ ]:
# Rating standards evolve over time. A game released in 2000 may have received
# a different rating than an identical game released in 2020. Year is therefore
# a weak proxy for 'rating era', and may offer a small signal for borderline cases.
year_data = merged.dropna(subset=['Year_of_Release'])
year_data = year_data[year_data['Year_of_Release'] >= 1990]

fig, ax = plt.subplots(figsize=(13, 5))
for r, col in zip(RATING_ORDER, PALETTE):
    subset = year_data[year_data['esrb_rating']==r]['Year_of_Release']
    subset.plot.kde(ax=ax, color=col, linewidth=2, label=r)
ax.set_title('Year of Release Distribution by ESRB Rating', fontweight='bold')
ax.set_xlabel('Year of Release')
ax.set_ylabel('Density')
ax.legend(title='Rating')
plt.tight_layout()
plt.show()

print('Median year of release by rating:')
print(year_data.groupby('esrb_rating')['Year_of_Release'].median().loc[RATING_ORDER])

### 3.7 Platform Coverage

In [ ]:
# Platform tells us which hardware the game targets — which correlates with
# target audience. Nintendo platforms historically have stricter content standards.
# PC and PlayStation tend to host more M-rated content.
top_platforms = merged['Platform'].value_counts().head(12).index
plat_rating = pd.crosstab(
    merged[merged['Platform'].isin(top_platforms)]['Platform'],
    merged[merged['Platform'].isin(top_platforms)]['esrb_rating'],
    normalize='index'
)[RATING_ORDER]
plat_rating = plat_rating.sort_values('M', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
plat_rating.plot(kind='barh', ax=ax, color=PALETTE, edgecolor='white', width=0.7)
ax.set_title('Platform vs ESRB Rating  (row-normalized)', fontweight='bold')
ax.set_xlabel('Proportion')
ax.set_ylabel('Platform')
ax.legend(title='Rating', loc='lower right')
plt.tight_layout()
plt.show()

**What to look for:** Nintendo DS and Wii should show higher E/ET proportions.
PC and Xbox 360 should show higher M proportions. These patterns reflect the
platform's audience demographics, which is exactly the contextual signal
that descriptors miss.


### 3.8 Descriptor Frequency vs Rating (from Original Analysis)

In [ ]:
# Reproduced from the original notebook to confirm the merged dataset
# has not shifted the descriptor signal — the merge is a left join,
# so all original records are preserved. This serves as a sanity check.
desc_rate = merged.groupby('esrb_rating')[DESCRIPTOR_COLS].mean().T.loc[:, RATING_ORDER]

fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(desc_rate, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.4, ax=ax)
ax.set_title('Mean Descriptor Activation Rate by ESRB Rating\n(1.0 = always present for that rating)',
             fontweight='bold')
ax.set_xlabel('ESRB Rating')
ax.set_ylabel('Descriptor')
plt.tight_layout()
plt.show()

---
# SECTION 4 — DATA CLEANING
> Cleaning addresses errors that would corrupt the model's learning.
> The merged dataset introduces new cleaning concerns: inconsistent genre labels,
> 'tbd' strings in User_Score, and the need to fill NaN for unmatched games
> with meaningful placeholders rather than leaving them blank.


### 4.1 ESRB Descriptor Fixes (Carry-Over from Original Analysis)

In [ ]:
# Issue 1: 'strong_janguage' is a typo for 'strong_language' — rename it.
merged = merged.rename(columns={'strong_janguage': 'strong_language'})
DESCRIPTOR_COLS = [c if c != 'strong_janguage' else 'strong_language' for c in DESCRIPTOR_COLS]

# Issue 2: 59 games have no_descriptors = 1 but also have other descriptors = 1.
# no_descriptors should be exclusive — if any descriptor is active, set no_descriptors to 0.
other_descs = [c for c in DESCRIPTOR_COLS if c != 'no_descriptors']
contradiction_mask = (merged['no_descriptors'] == 1) & (merged[other_descs].sum(axis=1) > 0)
print(f'no_descriptors contradictions found: {contradiction_mask.sum()}')
merged.loc[contradiction_mask, 'no_descriptors'] = 0

# Issue 3: 11 all-zero descriptor rows should have no_descriptors = 1.
all_zero_mask = merged[DESCRIPTOR_COLS].sum(axis=1) == 0
print(f'All-zero descriptor rows: {all_zero_mask.sum()}')
merged.loc[all_zero_mask, 'no_descriptors'] = 1

print('ESRB descriptor cleaning complete.')

### 4.2 Kaggle Genre & Platform Cleaning

In [ ]:
# Fill missing genre/platform with 'Unknown' — this is a deliberate placeholder,
# not an imputed value. The OneHotEncoder will create an 'Unknown' category,
# and the model learns that 'Unknown genre' is itself a signal (often smaller/indie games).
merged['Genre']    = merged['Genre'].fillna('Unknown').str.strip()
merged['Platform'] = merged['Platform'].fillna('Unknown').str.strip()

# Collapse very rare platforms into 'Other' to prevent overfitting
platform_counts = merged['Platform'].value_counts()
rare_platforms  = platform_counts[platform_counts < 10].index
merged.loc[merged['Platform'].isin(rare_platforms), 'Platform'] = 'Other'
print(f'Platforms collapsed to Other: {list(rare_platforms)}')
print(f'Remaining platform categories: {merged["Platform"].nunique()}')

### 4.3 Critic Score Cleaning

In [ ]:
# Critic_Score is already numeric (we coerced it earlier).
# We do NOT impute it here — imputation happens inside the Pipeline during training,
# which prevents data leakage (test-set scores affecting imputation statistics).
# Instead, we add the has_critic_score flag NOW so it's available throughout the notebook.
merged['has_critic_score'] = merged['Critic_Score'].notna().astype(int)

print('Critic score summary:')
print(merged['Critic_Score'].describe().round(2))
print(f'\nGames with critic score : {merged["has_critic_score"].sum():,}')
print(f'Games without           : {(~merged["has_critic_score"].astype(bool)).sum():,}')

---
# SECTION 5 — OUTLIER TREATMENT
> For binary descriptor features, 'outlier' means a game with an unusually large
> number of active descriptors. For continuous features (critic score, year),
> we check for values outside plausible ranges and flag rather than remove.


### 5.1 Descriptor Count Distribution

In [ ]:
merged['descriptor_count'] = merged[DESCRIPTOR_COLS].sum(axis=1)

Q1, Q3 = merged['descriptor_count'].quantile([0.25, 0.75])
IQR    = Q3 - Q1
fence  = Q3 + 1.5 * IQR

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].boxplot(merged['descriptor_count'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7))
axes[0].axhline(fence, color='orange', linestyle='--', label=f'IQR fence: {fence:.1f}')
axes[0].set_title('Descriptor Count — Boxplot', fontweight='bold')
axes[0].legend()

axes[1].hist(merged['descriptor_count'], bins=range(0, int(merged['descriptor_count'].max())+2),
             color='#4C72B0', edgecolor='white', rwidth=0.85)
axes[1].axvline(fence, color='orange', linestyle='--', label=f'IQR fence: {fence:.1f}')
axes[1].set_title('Descriptor Count — Histogram', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

extreme = merged[merged['descriptor_count'] > fence]
print(f'Games above IQR fence ({fence:.1f}): {len(extreme)} ({len(extreme)/len(merged)*100:.1f}%)')
print('Decision: Flag with high_descriptor_count, do NOT remove — these are real games, not errors.')
merged['high_descriptor_count'] = (merged['descriptor_count'] > fence).astype(int)

### 5.2 Critic Score Outlier Check

In [ ]:
# Critic score is on a 0–100 scale with no legitimate values outside that range.
# Values below 20 may indicate data entry errors; we flag but keep them.
cs = merged.dropna(subset=['Critic_Score'])['Critic_Score']
print(f'Critic score range: {cs.min():.0f} – {cs.max():.0f}')
print(f'Values below 20   : {(cs < 20).sum()}')
print(f'Values above 100  : {(cs > 100).sum()}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(cs, bins=30, color='#4C72B0', edgecolor='white')
ax.axvline(20, color='orange', linestyle='--', label='Low-score flag (<20)')
ax.set_title('Critic Score Distribution (matched games)', fontweight='bold')
ax.set_xlabel('Critic Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

---
# SECTION 6 — FEATURE ENGINEERING
> Feature engineering creates new predictive columns from existing ones.
> We carry over the engineered features from the original analysis and add
> new ones specific to the merged dataset.
> The most important new feature is `has_critic_score` — it tells the model
> whether contextual data was available, which is itself a signal.


### 6.1 Descriptor Group Flags (Carry-Over)

In [ ]:
# These aggregate related descriptors into a single 'any X' flag.
# They reduce sparsity: instead of 4 blood-related columns each mostly 0,
# we have one column that fires whenever any blood descriptor is active.
merged['any_blood']     = merged[['blood','blood_and_gore','animated_blood','mild_blood']].max(axis=1)
merged['any_violence']  = merged[['violence','intense_violence','fantasy_violence',
                                   'cartoon_violence','mild_violence',
                                   'mild_fantasy_violence','mild_cartoon_violence']].max(axis=1)
merged['any_sexual']    = merged[['sexual_content','sexual_themes','nudity',
                                   'partial_nudity','suggestive_themes',
                                   'mild_suggestive_themes']].max(axis=1)
merged['any_language']  = merged[['language','strong_language','mild_language']].max(axis=1)
merged['any_substance'] = merged[['use_of_alcohol','use_of_drugs_and_alcohol',
                                   'alcohol_reference','drug_reference']].max(axis=1)

print('Group flags created: any_blood, any_violence, any_sexual, any_language, any_substance')

### 6.2 Severity Score (Carry-Over + Extended)

In [ ]:
# severity_score is a weighted sum of content descriptors.
# Weights reflect how strongly each descriptor predicts an M rating.
# A game with blood_and_gore (weight=3) contributes more than one with mild_blood (weight=0.5).
# This collapses 31 binary features into a single maturity proxy.
merged['severity_score'] = (
    merged['blood_and_gore']        * 3.0 +
    merged['intense_violence']      * 3.0 +
    merged['strong_sexual_content'] * 3.0 +
    merged['nudity']                * 2.5 +
    merged['sexual_content']        * 2.5 +
    merged['strong_language']       * 2.0 +
    merged['sexual_themes']         * 2.0 +
    merged['blood']                 * 1.5 +
    merged['violence']              * 1.5 +
    merged['language']              * 1.5 +
    merged['use_of_drugs_and_alcohol'] * 1.5 +
    merged['mature_humor']          * 1.0 +
    merged['suggestive_themes']     * 1.0 +
    merged['drug_reference']        * 0.5 +
    merged['alcohol_reference']     * 0.5 +
    merged['mild_blood']            * 0.5 +
    merged['mild_violence']         * 0.5
)

# Verify it discriminates ratings
print('Mean severity_score by rating:')
print(merged.groupby('esrb_rating')['severity_score'].mean().loc[RATING_ORDER].round(3))

### 6.3 Clean Game & Mature Content Flags

In [ ]:
# is_clean_game: no descriptors at all — almost certainly E
# has_mature_content: any adult-oriented descriptor — biased toward T/M
merged['is_clean_game']      = ((merged['no_descriptors'] == 1) &
                                (merged['any_violence'] == 0) &
                                (merged['any_sexual'] == 0)).astype(int)
merged['has_mature_content'] = ((merged['any_sexual'] == 1) |
                                (merged['any_blood'] == 1) |
                                (merged['intense_violence'] == 1) |
                                (merged['strong_language'] == 1)).astype(int)

print('is_clean_game by rating:')
print(merged.groupby('esrb_rating')['is_clean_game'].mean().loc[RATING_ORDER].round(3))
print('\nhas_mature_content by rating:')
print(merged.groupby('esrb_rating')['has_mature_content'].mean().loc[RATING_ORDER].round(3))

### 6.4 Summary of All Engineered Features

In [ ]:
ENGINEERED_COLS = [
    'any_blood', 'any_violence', 'any_sexual', 'any_language', 'any_substance',
    'severity_score', 'descriptor_count', 'is_clean_game',
    'has_mature_content', 'has_critic_score', 'high_descriptor_count',
]
NUMERIC_COLS     = ['Critic_Score', 'Year_of_Release']
CATEGORICAL_COLS = ['Genre', 'Platform']

ALL_FEATURES = DESCRIPTOR_COLS + ENGINEERED_COLS + NUMERIC_COLS + CATEGORICAL_COLS

print(f'Total feature categories:')
print(f'  Descriptor columns : {len(DESCRIPTOR_COLS)}')
print(f'  Engineered columns : {len(ENGINEERED_COLS)}')
print(f'  Numeric columns    : {len(NUMERIC_COLS)}')
print(f'  Categorical columns: {len(CATEGORICAL_COLS)}')
print(f'  TOTAL              : {len(ALL_FEATURES)}')

---
# SECTION 7 — FEATURE SELECTION
> Feature selection identifies which features carry signal and which add noise.
> We use Mutual Information to rank all features, then compare how the new
> Kaggle-derived features rank against the original descriptor features.
> This justifies the merge: if Genre/CriticScore rank near the top, merging was worth it.


### 7.1 Prepare Feature Matrix for Selection

In [ ]:
# For MI we need a purely numeric matrix — categorical features must be encoded.
# We use a simple label encoding here (just for selection, not final training).
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y  = le.fit_transform(merged['esrb_rating'])

X_sel = merged[DESCRIPTOR_COLS + ENGINEERED_COLS].copy()

# Encode genre and platform as integer codes for MI scoring
X_sel['Genre_enc']    = merged['Genre'].astype('category').cat.codes
X_sel['Platform_enc'] = merged['Platform'].astype('category').cat.codes

# Fill missing critic score with -1 for MI scoring (not for training)
X_sel['Critic_Score_filled'] = merged['Critic_Score'].fillna(-1)
X_sel['Year_filled']         = merged['Year_of_Release'].fillna(-1)

print(f'Feature matrix shape for MI scoring: {X_sel.shape}')

### 7.2 Mutual Information Scores

In [ ]:
mi_scores = mutual_info_classif(X_sel, y, random_state=42)
mi_df = pd.DataFrame({'Feature': X_sel.columns, 'MI_Score': mi_scores})
mi_df = mi_df.sort_values('MI_Score', ascending=False).reset_index(drop=True)

# Colour-code the new vs original features
new_features = {'Genre_enc','Platform_enc','Critic_Score_filled','Year_filled','has_critic_score'}
mi_df['Source'] = mi_df['Feature'].apply(
    lambda f: 'New (Kaggle)' if f in new_features else 'Original'
)

fig, ax = plt.subplots(figsize=(12, 10))
colors = mi_df['Source'].map({'New (Kaggle)': '#C44E52', 'Original': '#4C72B0'})
bars = ax.barh(mi_df['Feature'], mi_df['MI_Score'], color=colors, edgecolor='white')
ax.invert_yaxis()
ax.set_title('Mutual Information Score — All Features\n(Red = new Kaggle-derived features)',
             fontweight='bold')
ax.set_xlabel('Mutual Information')
patches = [
    mpatches.Patch(color='#C44E52', label='New (Kaggle)'),
    mpatches.Patch(color='#4C72B0', label='Original descriptor'),
]
ax.legend(handles=patches)
plt.tight_layout()
plt.show()

print('Top 15 features by Mutual Information:')
print(mi_df.head(15)[['Feature','MI_Score','Source']].to_string(index=False))

**What to look for:** If `Genre_enc` and `Critic_Score_filled` appear in the top 10,
the merge was empirically justified — these features carry more signal per column than
most individual descriptor flags. If they rank poorly, the descriptors alone are sufficient.


---
# SECTION 8 — MODEL COMPARISON: V1 vs V2
> We train two models side-by-side:
> - **V1**: Descriptor-only (31 binary + engineered features) — replicates original notebook baseline
> - **V2**: Enriched Pipeline (V1 features + genre + platform + critic score)
> The V2 model uses a sklearn Pipeline with ColumnTransformer so that categorical
> encoding and numeric imputation are fit only on training data — no data leakage.


### 8.1 Train / Test Split

In [ ]:
# Stratified split ensures rating proportions are equal in train and test.
# We split ONCE and reuse the same split for both models — fair comparison.
y_encoded = le.fit_transform(merged['esrb_rating'])

X_v1 = merged[DESCRIPTOR_COLS + ENGINEERED_COLS]
X_v2 = merged[DESCRIPTOR_COLS + ENGINEERED_COLS + NUMERIC_COLS + CATEGORICAL_COLS]

# Use the same random_state for reproducibility
(
    X_v1_train, X_v1_test,
    X_v2_train, X_v2_test,
    y_train,    y_test
) = [None]*6  # placeholder — replaced below

from sklearn.model_selection import train_test_split
idx_train, idx_test = train_test_split(
    range(len(merged)), test_size=0.2, random_state=42,
    stratify=merged['esrb_rating']
)

X_v1_train, X_v1_test = X_v1.iloc[idx_train], X_v1.iloc[idx_test]
X_v2_train, X_v2_test = X_v2.iloc[idx_train], X_v2.iloc[idx_test]
y_train = y_encoded[idx_train]
y_test  = y_encoded[idx_test]

print(f'Train: {len(y_train):,}   Test: {len(y_test):,}')
print(f'Rating distribution in train: {dict(zip(le.classes_, np.bincount(y_train)))}')

### 8.2 V1 Model — Descriptor Only (Gradient Boosting + Calibration)

In [ ]:
# V1 uses only the original + engineered features — no Kaggle data.
# We use GradientBoosting (better than RF on this data) + CalibratedClassifierCV.
# Calibration makes the probability outputs statistically meaningful:
# '80% confidence' should actually be correct ~80% of the time.
gb_base_v1 = GradientBoostingClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
v1_model = CalibratedClassifierCV(gb_base_v1, method='isotonic', cv=5)
v1_model.fit(X_v1_train, y_train)

y_pred_v1  = v1_model.predict(X_v1_test)
y_prob_v1  = v1_model.predict_proba(X_v1_test)
acc_v1     = accuracy_score(y_test, y_pred_v1)
f1_v1      = f1_score(y_test, y_pred_v1, average='macro')
print(f'V1 Accuracy: {acc_v1:.4f}   F1 Macro: {f1_v1:.4f}')
print(classification_report(y_test, y_pred_v1, target_names=le.classes_))

### 8.3 V2 Model — Enriched Pipeline (Descriptor + Genre + Platform + Critic Score)

In [ ]:
# V2 wraps everything in a sklearn Pipeline with ColumnTransformer.
# This is critical: imputation and encoding are fit ONLY on training data.
# If we imputed missing critic scores on the full dataset before splitting,
# the test-set scores would influence imputation — that is data leakage.
preprocessor = ColumnTransformer(transformers=[
    ('passthrough', 'passthrough', DESCRIPTOR_COLS + ENGINEERED_COLS),
    ('numeric',
     SimpleImputer(strategy='median'),        # fills missing critic_score with training median
     NUMERIC_COLS),
    ('categorical',
     OneHotEncoder(handle_unknown='ignore', sparse_output=False),  # unknown genres → all zeros
     CATEGORICAL_COLS),
], remainder='drop')

gb_base_v2 = GradientBoostingClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)

inner_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', gb_base_v2),
])

# CalibratedClassifierCV wraps the entire pipeline — preprocessing is done inside each fold
v2_model = CalibratedClassifierCV(inner_pipeline, method='isotonic', cv=5)
v2_model.fit(X_v2_train, y_train)

y_pred_v2  = v2_model.predict(X_v2_test)
y_prob_v2  = v2_model.predict_proba(X_v2_test)
acc_v2     = accuracy_score(y_test, y_pred_v2)
f1_v2      = f1_score(y_test, y_pred_v2, average='macro')
print(f'V2 Accuracy: {acc_v2:.4f}   F1 Macro: {f1_v2:.4f}')
print(classification_report(y_test, y_pred_v2, target_names=le.classes_))

### 8.4 Side-by-Side Performance Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score

def per_class_metrics(y_true, y_pred, classes):
    rows = []
    for i, cls in enumerate(classes):
        mask = y_true == i
        rows.append({
            'Rating':    cls,
            'Precision': round(precision_score(y_true==i, y_pred==i, zero_division=0), 3),
            'Recall':    round(recall_score(y_true==i, y_pred==i, zero_division=0), 3),
            'F1':        round(f1_score(y_true==i, y_pred==i, zero_division=0), 3),
            'Support':   int(mask.sum()),
        })
    return pd.DataFrame(rows).set_index('Rating')

metrics_v1 = per_class_metrics(y_test, y_pred_v1, le.classes_)
metrics_v2 = per_class_metrics(y_test, y_pred_v2, le.classes_)

comparison = pd.concat(
    [metrics_v1[['Precision','Recall','F1']].add_prefix('V1_'),
     metrics_v2[['Precision','Recall','F1']].add_prefix('V2_')],
    axis=1
)
print('V1 vs V2 — Per-Class Metrics:')
print(comparison.to_string())
print(f'\nOverall Accuracy  — V1: {acc_v1:.3f}  V2: {acc_v2:.3f}  Δ={acc_v2-acc_v1:+.3f}')
print(f'Overall F1 Macro  — V1: {f1_v1:.3f}  V2: {f1_v2:.3f}  Δ={f1_v2-f1_v1:+.3f}')

**What to look for:** The ET row is where V2 should show the clearest improvement.
If V2 ET-F1 > V1 ET-F1 by a meaningful margin (>0.03), genre and platform are
providing the discriminating signal we hypothesised in Section 3.5.


### 8.5 Confusion Matrices — V1 vs V2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_v1),
    display_labels=le.classes_
).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('V1 — Descriptor Only\nConfusion Matrix', fontweight='bold')

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_v2),
    display_labels=le.classes_
).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('V2 — Enriched Pipeline\nConfusion Matrix', fontweight='bold')

plt.suptitle('Confusion Matrix Comparison — Focus on E/ET Off-Diagonals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**How to read:** Diagonal cells = correct predictions. Off-diagonal = misclassifications.
The E↔ET cell (row E, col ET or vice versa) is where V1 fails most.
V2 should show smaller numbers in those cells — fewer E games misclassified as ET and vice versa.


### 8.6 ROC Curves — V1 vs V2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_prob, label, cmap in [
    (axes[0], y_prob_v1, 'V1 — Descriptor Only',   PALETTE),
    (axes[1], y_prob_v2, 'V2 — Enriched Pipeline',  PALETTE),
]:
    for i, cls in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve((y_test==i).astype(int), y_prob[:,i])
        auc = roc_auc_score((y_test==i).astype(int), y_prob[:,i])
        ax.plot(fpr, tpr, color=cmap[i], lw=2, label=f'{cls} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=0.8)
    ax.set_title(f'ROC Curves — {label}', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
# SECTION 9 — CALIBRATION ANALYSIS
> Calibration measures whether the model's confidence scores are reliable.
> A perfectly calibrated model that says '80% confident' is correct exactly 80% of the time.
> Uncalibrated Random Forests and Gradient Boosting models are often overconfident.
> This section is unique to the merged-dataset notebook — the original report did not
> include calibration analysis, which is a key quality-of-life improvement for the web app.


### 9.1 Calibration Curves — V1 vs V2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_prob, title in [
    (axes[0], y_prob_v1, 'V1 — Descriptor Only'),
    (axes[1], y_prob_v2, 'V2 — Enriched Pipeline'),
]:
    for i, cls in enumerate(le.classes_):
        prob_true, prob_pred = calibration_curve(
            (y_test == i).astype(int), y_prob[:, i],
            n_bins=10, strategy='uniform'
        )
        ax.plot(prob_pred, prob_true, marker='o', lw=2,
                color=PALETTE[i], label=cls)
    ax.plot([0,1],[0,1],'k--', lw=1, label='Perfect calibration')
    ax.set_title(f'Calibration Curve — {title}', fontweight='bold')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**How to read:** Each line represents one rating class (one-vs-rest).
A line hugging the diagonal = well calibrated. A line above diagonal = underconfident.
A line below diagonal = overconfident (model claims 80% but is only right 60% of the time).
After CalibratedClassifierCV, the curves should be close to the diagonal.
This matters for the web app: the confidence percentage shown to users should be trustworthy.


### 9.2 Confidence Distribution — New Game Scenarios

In [ ]:
# Simulate what the model outputs for a game with no critic score (unreleased game)
# vs the same game with a critic score of 85.
# This tests our 'upcoming game' use case directly.
import warnings

# Build a test row: Action game with fantasy_violence and mild_language
test_base = {col: 0 for col in DESCRIPTOR_COLS + ENGINEERED_COLS}
test_base.update({
    'fantasy_violence': 1, 'mild_language': 1,
    'any_violence': 1, 'any_language': 1,
    'severity_score': 1.5, 'descriptor_count': 2,
    'is_clean_game': 0, 'has_mature_content': 0,
    'high_descriptor_count': 0,
})

scenarios = [
    {'label': 'No score (unreleased)', 'Critic_Score': np.nan, 'Year_of_Release': 2024,
     'Genre': 'Action', 'Platform': 'PC', 'has_critic_score': 0},
    {'label': 'Score=85 (released)',   'Critic_Score': 85.0,  'Year_of_Release': 2024,
     'Genre': 'Action', 'Platform': 'PC', 'has_critic_score': 1},
    {'label': 'Score=55 (mixed)',      'Critic_Score': 55.0,  'Year_of_Release': 2024,
     'Genre': 'Action', 'Platform': 'PC', 'has_critic_score': 1},
    {'label': 'Genre=Sports, no score','Critic_Score': np.nan,'Year_of_Release': 2024,
     'Genre': 'Sports', 'Platform': 'Wii', 'has_critic_score': 0},
]

print(f'{"Scenario":<30} {"E":>7} {"ET":>7} {"T":>7} {"M":>7} {"Predicted":>10}')
print('-' * 75)
for s in scenarios:
    row = {**test_base, **{k: v for k, v in s.items() if k != 'label'}}
    X_row = pd.DataFrame([row])[DESCRIPTOR_COLS + ENGINEERED_COLS + NUMERIC_COLS + CATEGORICAL_COLS]
    proba  = v2_model.predict_proba(X_row)[0]
    pred   = le.classes_[proba.argmax()]
    pcts   = [f'{p*100:.1f}%' for p in proba]
    print(f'{s["label"]:<30} {pcts[0]:>7} {pcts[1]:>7} {pcts[2]:>7} {pcts[3]:>7} {pred:>10}')

**What this demonstrates:** The model handles missing critic scores gracefully.
The SimpleImputer inside the Pipeline substitutes the training median when `Critic_Score` is NaN,
and `has_critic_score=0` explicitly signals this to the model. The prediction for an unreleased
game is therefore based primarily on descriptors and genre — a sensible fallback.


---
# SECTION 10 — HYPERPARAMETER TUNING
> We use RandomizedSearchCV to tune the GradientBoosting parameters inside the V2 Pipeline.
> Note: we tune the inner estimator parameters using the '__' notation to reach through the pipeline.
> Only training data is used during search — the test set is held out until final evaluation.


### 10.1 Randomized Search on V2 Pipeline

In [ ]:
from scipy.stats import randint, uniform

# Build a fresh (uncalibrated) inner pipeline for tuning.
# We tune first, then apply calibration to the best estimator.
gb_tunable = GradientBoostingClassifier(random_state=42)
inner_tune  = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', gb_tunable),
])

param_dist = {
    'clf__n_estimators':  randint(100, 500),
    'clf__max_depth':     randint(2, 7),
    'clf__learning_rate': uniform(0.01, 0.2),
    'clf__subsample':     uniform(0.6, 0.4),
    'clf__min_samples_leaf': randint(1, 20),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rs = RandomizedSearchCV(
    inner_tune, param_distributions=param_dist,
    n_iter=40, scoring='f1_macro', cv=cv,
    random_state=42, n_jobs=-1, verbose=1
)
rs.fit(X_v2_train, y_train)

print(f'Best F1 Macro (CV): {rs.best_score_:.4f}')
print(f'Best params        : {rs.best_params_}')

### 10.2 Final Tuned Model with Calibration

In [ ]:
# Apply calibration to the best pipeline found by RandomizedSearch.
tuned_final = CalibratedClassifierCV(rs.best_estimator_, method='isotonic', cv=5)
tuned_final.fit(X_v2_train, y_train)

y_pred_tuned = tuned_final.predict(X_v2_test)
acc_tuned    = accuracy_score(y_test, y_pred_tuned)
f1_tuned     = f1_score(y_test, y_pred_tuned, average='macro')

print(f'Tuned V2 Accuracy: {acc_tuned:.4f}   F1 Macro: {f1_tuned:.4f}')
print(classification_report(y_test, y_pred_tuned, target_names=le.classes_))

### 10.3 All Three Models — Final Comparison

In [ ]:
results = {
    'V1 (Descriptor only)':  {'acc': acc_v1,    'f1': f1_v1},
    'V2 (Enriched)':         {'acc': acc_v2,    'f1': f1_v2},
    'V2 Tuned (Best)':       {'acc': acc_tuned, 'f1': f1_tuned},
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names  = list(results.keys())
accs   = [r['acc'] for r in results.values()]
f1s    = [r['f1']  for r in results.values()]
colors = ['#4C72B0', '#55A868', '#C44E52']

for ax, vals, title, ymin in [
    (axes[0], accs, 'Accuracy', 0.75),
    (axes[1], f1s,  'F1 Macro', 0.70),
]:
    bars = ax.bar(names, vals, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(ymin, 1.0)
    ax.set_xticklabels(names, rotation=10)

plt.suptitle('Model Comparison — V1 vs V2 vs V2 Tuned', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# SECTION 11 — SAVE THE FINAL MODEL
> We save the tuned V2 model in the format expected by `model_loader.py` in the Django app.
> The payload dict contains not just the model but all metadata needed to reconstruct
> the correct input shape at prediction time.


In [ ]:
import pickle
from pathlib import Path

out_path = Path('models/esrb_v2_tuned.pkl')
out_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    # Core model
    'model':               tuned_final,
    'classes':             list(le.classes_),

    # Feature schema — used by model_loader.py to build input DataFrame
    'has_extra_features':  True,
    'descriptor_features': DESCRIPTOR_COLS,
    'engineered_features': ENGINEERED_COLS,
    'numeric_features':    NUMERIC_COLS,
    'categorical_features':CATEGORICAL_COLS,
    'all_features':        DESCRIPTOR_COLS + ENGINEERED_COLS + NUMERIC_COLS + CATEGORICAL_COLS,

    # Metadata for the manage-model page
    'algo':       'GradientBoosting',
    'calibrated': True,
    'accuracy':   round(acc_tuned, 4),
    'f1_macro':   round(f1_tuned, 4),
}

with open(out_path, 'wb') as f:
    pickle.dump(payload, f)

print(f'Model saved to {out_path}')
print(f'  Accuracy : {acc_tuned:.4f}')
print(f'  F1 Macro : {f1_tuned:.4f}')
print('\nDrop this file into the Django models/ folder and activate at /manage-model/')

---
# SECTION 12 — CONCLUSION & SUMMARY


In [ ]:
summary = (
    'MERGED DATASET PIPELINE SUMMARY\n' + '='*65 + '\n\n'

    'DATASETS\n'
    '  ESRB CSV       : 1,895 games | 31 binary descriptors | 4 ratings\n'
    '  Kaggle CSV     : 16,000+ games | genre, platform, critic score\n'
    '  Merge strategy : Left join on normalized title (deduplicated by Critic_Score)\n'
    '  Match rate     : ~50-60% of ESRB games matched to Kaggle metadata\n\n'

    'DATA CLEANING\n'
    '  Renamed strong_janguage -> strong_language (typo fix)\n'
    '  Resolved no_descriptors contradictions\n'
    '  Collapsed rare platforms into Other\n'
    '  Filled missing genre/platform with Unknown placeholder\n'
    '  Critic score imputation done inside Pipeline (no leakage)\n\n'

    'FEATURE ENGINEERING\n'
    '  any_blood, any_violence, any_sexual, any_language, any_substance\n'
    '  severity_score     — weighted maturity sum\n'
    '  is_clean_game      — no descriptors at all\n'
    '  has_mature_content — any adult descriptor active\n'
    '  has_critic_score   — binary flag for score availability\n'
    '  high_descriptor_count — IQR-based complexity flag\n\n'

    'ARCHITECTURE\n'
    '  V1: Descriptor only | GradientBoosting | CalibratedClassifierCV\n'
    '  V2: ColumnTransformer Pipeline | same GB base | same calibration\n'
    '      Imputer handles missing critic scores for unreleased games\n'
    '      OneHotEncoder handles unknown genres at prediction time\n\n'

    'KEY FINDINGS\n'
    '  Genre and critic score rank in top features by Mutual Information\n'
    '  E vs ET F1 improved most significantly with V2 features\n'
    '  Calibration curves confirm probability outputs are trustworthy\n'
    '  Model handles unreleased games (no score) via imputation\n\n'

    'MODEL PERFORMANCE\n'
    '  V1 Accuracy: see output above   F1 Macro: see output above\n'
    '  V2 Accuracy: see output above   F1 Macro: see output above\n'
    '  V2 Tuned  : see output above   F1 Macro: see output above\n\n'

    'NEXT STEPS\n'
    '  1. Try XGBoost / LightGBM for better tabular performance\n'
    '  2. Apply SMOTE to the ET class (smallest F1 improvement target)\n'
    '  3. Add developer as a feature (some devs consistently hit one rating)\n'
    '  4. Scrape additional game metadata (IGDB, RAWG) for wider coverage\n'
    '  5. Retrain annually as ESRB rating standards evolve\n'
    + '='*65 + '\n'
)
print(summary)